# Lesson 4 · Retrieval on its own: a nearest-neighbour detector

<a href="https://colab.research.google.com/github/Mary-Vadi/rag/blob/main/lessons/04_retrieval_as_a_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In this lesson you will**

- turn a text's neighbours into a prediction: a **vote**
- measure a detector properly with accuracy, precision, recall, F1 and the confusion matrix
- choose settings on a **validation** set and report the **test** result only once
- see how the number of neighbours `k` and the voting weights change the results

**Runtime:** a CPU works; a GPU makes the embedding step faster.

## 1. From neighbours to a prediction

In lesson 3 we found each text's nearest neighbours. Now we let them vote.

**Plain vote.** `P(machine)` is simply the share of neighbours that are machine-written. With `k = 7` neighbours labelled `[1, 1, 0, 1, 0, 1, 1]`:

$$P(\text{machine}) = \frac{5}{7} \approx 0.71 \;\ge\; 0.5 \;\Rightarrow\; \text{predict machine}$$

**Weighted vote.** Maybe the closest neighbours deserve a bigger say than the 7th closest. We give each neighbour a weight based on its similarity $s$, using a **softmax** with a **temperature** $T$:

$$w_i = \frac{e^{s_i / T}}{\sum_j e^{s_j / T}} \qquad P(\text{machine}) = \sum_i w_i \cdot \text{label}_i$$

A small $T$ makes the closest neighbour dominate; a large $T$ makes the vote almost equal again. Let's see that with made-up numbers.

## 2. Setup

In [ ]:
# Setup: run this cell first.
# In Colab it downloads the course code and installs the extra libraries.
# On your own computer (from the lessons/ folder) it just makes the code importable.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("/content/rag"):
        !git clone -q https://github.com/Mary-Vadi/rag.git /content/rag
        !pip install -q -r /content/rag/requirements.txt
    %cd /content/rag
elif os.path.basename(os.getcwd()) == "lessons":
    %cd ..
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ragdetect.data import load_mage
from ragdetect.embeddings import Embedder
from ragdetect.evaluation import compare, evaluate, plot_confusion
from ragdetect.retrieval import RetrievalDetector, neighbour_vote

In [ ]:
similarities = np.array([[0.83, 0.81, 0.80, 0.78, 0.77, 0.75, 0.74]])
labels = np.array([[0, 1, 1, 1, 0, 1, 0]])

for temperature in [None, 1.0, 0.1, 0.01]:
    p = neighbour_vote(similarities, labels, temperature)[0]
    print(f"temperature {str(temperature):>5}: P(machine) = {p:.3f}")

Notice that `temperature = 1.0` gives almost the same answer as the plain vote. The similarities are all close together (0.74 to 0.83), and dividing by 1 doesn't spread them apart, so every neighbour ends up with nearly the same weight. **The original experiment used exactly this setting**, so in practice it was a plain majority vote. Only a much smaller temperature lets the nearest neighbour take over.

## 3. Three separate sets

We now need **three** sets of MAGE texts, and each has one job:

| Set | MAGE split | Size | Job |
|---|---|---|---|
| pool | train | 5,000 | the labelled examples we retrieve from |
| validation | validation | 1,000 | trying out settings (`k`, temperature) |
| test | test | 2,000 | the final score, measured **once** at the very end |

Why not choose `k` by looking at the test score? Because then the test score stops being an honest estimate: we'd have picked whatever happened to work best on those particular texts. This mistake is common and easy to make, so we separate the sets from the start.

In [ ]:
pool = load_mage("train", n=5000)
validation = load_mage("validation", n=1000)
test = load_mage("test", n=2000)

embedder = Embedder()
pool_vectors = embedder.encode(pool["text"], show_progress=True)
validation_vectors = embedder.encode(validation["text"], show_progress=True)
test_vectors = embedder.encode(test["text"], show_progress=True)

## 4. How do we measure a detector?

Let's build a first detector (plain vote, `k = 7`) and score it on the validation set.

In [ ]:
detector = RetrievalDetector(k=7, temperature=None).fit(pool_vectors, pool["label"])
validation_proba = detector.predict_proba(validation_vectors)

plot_confusion(validation["label"], validation_proba, title="Retrieval only, k = 7 (validation)")
plt.show()
evaluate(validation["label"], validation_proba)

### Reading the confusion matrix

Rows are the **true** label, columns are the **predicted** label:

| | predicted human | predicted machine |
|---|---|---|
| **really human** | correct ✅ | *false alarm*: a person accused of using AI ❌ |
| **really machine** | *missed*: AI text that slipped through ❌ | correct ✅ |

### The four numbers

We treat "machine" as the **positive** class (the thing we're trying to catch).

- **Accuracy**: the share of all texts labelled correctly. Easy to understand, but it hides *which* mistakes are being made.
- **Precision**: of the texts flagged as machine, how many really were? Low precision means many false alarms, which is a serious problem if the detector is used to accuse students of cheating.
- **Recall**: of the machine texts, how many did we catch? Low recall means AI text slips through.
- **F1**: a single number that balances precision and recall (their harmonic mean). It is only high when *both* are.

A detector that labels *everything* as machine gets 100% recall and looks great on that one number, while being useless. Always look at several metrics and at the confusion matrix.

## 5. Choosing k (on the validation set)

In [ ]:
rows = []
for k in [1, 3, 5, 7, 15, 31, 51, 101]:
    proba = RetrievalDetector(k=k, temperature=None).fit(pool_vectors, pool["label"]).predict_proba(validation_vectors)
    rows.append({"k": k, **{m: v for m, v in evaluate(validation["label"], proba).items() if m != "confusion"}})
by_k = pd.DataFrame(rows).set_index("k")

by_k[["accuracy", "f1"]].plot(marker="o", logx=True, figsize=(7, 3.5), title="Validation score by number of neighbours k")
plt.show()
by_k.round(3)

Very small `k` trusts one or two neighbours completely, so a single odd neighbour can flip the answer. Very large `k` averages over so many texts that the vote drifts towards the overall balance of the pool (50/50). The best value is usually somewhere in between.

## 6. Plain or weighted vote?

Using the best `k` from above, compare a few temperatures, still on the validation set.

In [ ]:
best_k = int(by_k["accuracy"].idxmax())
rows = []
for temperature in [None, 1.0, 0.1, 0.05, 0.02]:
    proba = RetrievalDetector(k=best_k, temperature=temperature).fit(pool_vectors, pool["label"]).predict_proba(validation_vectors)
    rows.append({"temperature": str(temperature), **{m: v for m, v in evaluate(validation["label"], proba).items() if m != "confusion"}})
by_temperature = pd.DataFrame(rows).set_index("temperature")
print("k =", best_k)
by_temperature.round(3)

## 7. The final test score

Now, and only now, we pick the best settings from the validation set and score them **once** on the test set.

In [ ]:
best_temperature = by_temperature["accuracy"].idxmax()
best_temperature = None if best_temperature == "None" else float(best_temperature)
print(f"chosen settings: k = {best_k}, temperature = {best_temperature}")

final = RetrievalDetector(k=best_k, temperature=best_temperature).fit(pool_vectors, pool["label"])
test_proba = final.predict_proba(test_vectors)

plot_confusion(test["label"], test_proba, title="Retrieval only (test)")
plt.show()
compare({"retrieval only (MiniLM)": evaluate(test["label"], test_proba)})

**Write this result down.** In lesson 6 we'll compare it with a trained classifier and with the combination of the two.

## Check your understanding

<details>
<summary>1. A detector has 95% recall but 55% precision. What is it doing?</summary>

It catches almost all machine texts, but nearly half of the texts it flags are actually human. It's "trigger-happy": it labels too much as machine. In a classroom that would mean many students wrongly accused.
</details>

<details>
<summary>2. What happens if k equals the size of the whole pool?</summary>

Every query gets the same neighbours (all of them), so every query gets the same score: the share of machine texts in the pool, 0.5 here. The detector can no longer tell texts apart.
</details>

<details>
<summary>3. Why not choose k by looking at the test set?</summary>

Then the test score would no longer be an honest estimate of performance on new data, because we'd have tuned our choices to those exact texts. Settings are chosen on validation data, and the test set is used once at the end.
</details>

## Try it yourself

1. Make the pool smaller (500 texts) or larger (20,000). How does the test accuracy change? What does that tell you about the value of collecting labelled examples?
2. Look at the texts retrieval gets wrong: `test[(test_proba >= 0.5) != test["label"]]`. Do they come from particular sources?

## Summary

- A retrieval detector predicts by letting a text's `k` nearest neighbours vote.
- Weighted voting with a temperature lets closer neighbours count more; with cosine similarities, `T = 1` is almost a plain vote.
- Look at accuracy, precision, recall, F1 **and** the confusion matrix.
- Choose settings on the validation set; report the test set once.

**Next:** [Lesson 5 · Training a classifier](05_training_a_classifier.ipynb), where we train the other half of our detector and meet domain shift.